<a href="https://colab.research.google.com/github/ntomben97/NYC-Yellow-Taxi-Big-Data-Project/blob/main/Download_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Checking the original Data

In [ ]:
import requests
import pandas as pd

files = []

for year in range(2009, 2027):

    for month in range(1, 13):

        month_str = f"{month:02d}"

        url = (
            f"https://d37ci6vzurychx.cloudfront.net/"
            f"trip-data/yellow_tripdata_{year}-{month_str}.parquet"
        )

        try:
            response = requests.head(
                url,
                allow_redirects=True,
                timeout=10
            )

            # Print the response for newer years
            if year >= 2022:
                print(
                    f"{year}-{month_str}: "
                    f"HTTP {response.status_code} | "
                    f"Size: {response.headers.get('Content-Length', 'N/A')}"
                )

            if response.status_code == 200:

                size_bytes = int(
                    response.headers.get("Content-Length", 0)
                )

                if size_bytes > 0:

                    files.append({
                        "Year": year,
                        "Month": month,
                        "File": f"yellow_tripdata_{year}-{month_str}.parquet",
                        "Size_GB": size_bytes / (1024 ** 3),
                        "Size_Bytes": size_bytes
                    })

        except requests.RequestException as e:
            print(f"{year}-{month_str}: ERROR - {e}")

file_sizes = pd.DataFrame(files)

file_sizes



total_size_gb = file_sizes["Size_GB"].sum()

print(f"Number of files found: {len(file_sizes)}")
print(f"Total Yellow Taxi dataset size: {total_size_gb:.2f} GB")

2022-01: HTTP 200 | Size: 38139949
2022-02: HTTP 200 | Size: 45616512
2022-03: HTTP 200 | Size: 55682369
2022-04: HTTP 200 | Size: 55222692
2022-05: HTTP 200 | Size: 55558821
2022-06: HTTP 200 | Size: 55365184
2022-07: HTTP 200 | Size: 49367712
2022-08: HTTP 200 | Size: 49717159
2022-09: HTTP 200 | Size: 49619957
2022-10: HTTP 200 | Size: 57061938
2022-11: HTTP 200 | Size: 50106631
2022-12: HTTP 200 | Size: 53640739
2023-01: HTTP 200 | Size: 47673370
2023-02: HTTP 200 | Size: 47748012
2023-03: HTTP 200 | Size: 56127762
2023-04: HTTP 200 | Size: 54222699
2023-05: HTTP 200 | Size: 58654627
2023-06: HTTP 200 | Size: 54999465
2023-07: HTTP 200 | Size: 48361828
2023-08: HTTP 200 | Size: 48152353
2023-09: HTTP 200 | Size: 47895515
2023-10: HTTP 200 | Size: 59009059
2023-11: HTTP 200 | Size: 56094653
2023-12: HTTP 200 | Size: 56804275
2024-01: HTTP 200 | Size: 49961641
2024-02: HTTP 200 | Size: 50349284
2024-03: HTTP 200 | Size: 60078280
2024-04: HTTP 200 | Size: 59133625
2024-05: HTTP 200 | 

In [ ]:
yearly_sizes = (
    file_sizes
    .groupby("Year")["Size_GB"]
    .sum()
    .reset_index()
)

yearly_sizes.sort_values("Year", ascending=False)

,Year,Size_GB
17,2026,0.302919
16,2025,0.772973
15,2024,0.645408
14,2023,0.592082
13,2022,0.572856
12,2021,0.446644
11,2020,0.348873
10,2019,1.158130
9,2018,1.363844
8,2017,1.485167


In [ ]:
recent_years = yearly_sizes.sort_values("Year", ascending=False).copy()

recent_years["Cumulative_GB"] = recent_years["Size_GB"].cumsum()

recent_years


,Year,Size_GB,Cumulative_GB
17,2026,0.302919,0.302919
16,2025,0.772973,1.075892
15,2024,0.645408,1.721300
14,2023,0.592082,2.313383
13,2022,0.572856,2.886239
12,2021,0.446644,3.332883
11,2020,0.348873,3.681756
10,2019,1.158130,4.839886
9,2018,1.363844,6.203731
8,2017,1.485167,7.688898


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import requests
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

BASE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data"

DOWNLOAD_DIR = Path(
    "/content/drive/MyDrive/MIT805/Group Project/src"
)

DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

Mounted at /content/drive


In [ ]:

def download_file(year, month):
    month_str = f"{month:02d}"
    filename = f"yellow_tripdata_{year}-{month_str}.parquet"
    url = f"{BASE_URL}/{filename}"
    output_file = DOWNLOAD_DIR / filename

    if output_file.exists() and output_file.stat().st_size > 0:
        return f"Already exists: {filename}"

    try:
        response = requests.get(
            url,
            stream=True,
            timeout=(30, 300)
        )

        if response.status_code == 200:

            with open(output_file, "wb") as f:
                for chunk in response.iter_content(
                    chunk_size=8 * 1024 * 1024
                ):
                    if chunk:
                        f.write(chunk)

            size_mb = output_file.stat().st_size / (1024**2)

            return f"✓ {filename} ({size_mb:.1f} MB)"

        elif response.status_code == 404:
            return f"Not available: {filename}"

        else:
            return f"HTTP {response.status_code}: {filename}"

    except Exception as e:
        return f"ERROR {filename}: {e}"


# Create list of files to download
tasks = [
    (year, month)
    for year in range(2014, 2027)
    for month in range(1, 13)
]

# Download 4 files at a time
with ThreadPoolExecutor(max_workers=4) as executor:

    futures = [
        executor.submit(download_file, year, month)
        for year, month in tasks
    ]

    for future in as_completed(futures):
        print(future.result())

✓ yellow_tripdata_2014-01.parquet (164.0 MB)
✓ yellow_tripdata_2014-02.parquet (154.7 MB)
✓ yellow_tripdata_2014-04.parquet (174.7 MB)
✓ yellow_tripdata_2014-03.parquet (183.5 MB)
✓ yellow_tripdata_2014-07.parquet (157.5 MB)
✓ yellow_tripdata_2014-05.parquet (178.0 MB)
✓ yellow_tripdata_2014-08.parquet (165.2 MB)
✓ yellow_tripdata_2014-06.parquet (165.8 MB)
✓ yellow_tripdata_2014-09.parquet (175.3 MB)
✓ yellow_tripdata_2014-11.parquet (172.7 MB)
✓ yellow_tripdata_2014-12.parquet (170.7 MB)
✓ yellow_tripdata_2014-10.parquet (186.6 MB)
✓ yellow_tripdata_2015-02.parquet (163.7 MB)
✓ yellow_tripdata_2015-04.parquet (172.1 MB)
✓ yellow_tripdata_2015-03.parquet (176.8 MB)
✓ yellow_tripdata_2015-01.parquet (167.2 MB)
✓ yellow_tripdata_2015-06.parquet (163.8 MB)
✓ yellow_tripdata_2015-07.parquet (153.2 MB)
✓ yellow_tripdata_2015-08.parquet (147.1 MB)
✓ yellow_tripdata_2015-05.parquet (174.0 MB)
✓ yellow_tripdata_2015-10.parquet (163.2 MB)
✓ yellow_tripdata_2015-12.parquet (152.5 MB)
✓ yellow_t